# Splitify — Multi-stem training (Google Colab)

This notebook trains the **multi-stem** U-Net in `learning/multistem` using the preprocessed HDF5 dataset.

## Prereqs
- Colab runtime with **GPU** enabled (`Runtime → Change runtime type → GPU`).
- `data.zip` uploaded to your Google Drive (should contain `hdf5s/` and `indexes/`).
- The Splitify code in a Git repo you can clone.

## Notes
- We extract data to Colab local disk (`/content`) for faster I/O.
- We regenerate `indexes/*.pkl` on Colab so paths match the Colab filesystem.

In [2]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


torch: 2.10.0+cpu
cuda available: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
# Fill these in
REPO_URL = "https://github.com/rotemarie/splitify"  # e.g. https://github.com/you/splitify.git
REPO_DIR = "/content/splitify"

# Path to your uploaded data.zip in Google Drive
DATA_ZIP_IN_DRIVE = "/content/drive/MyDrive/splitify/data"


In [5]:
!rm -rf "{REPO_DIR}"
!git clone "{REPO_URL}" "{REPO_DIR}"
%cd "{REPO_DIR}"


Cloning into '/content/splitify'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 37 (delta 7), reused 32 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 27.23 KiB | 6.81 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/splitify


In [6]:
!python -m pip install -U pip
# Some environments fail to build/install pickle5; fallback install keeps training unblocked.
!pip install -r requirements.txt || pip install torch torchaudio numpy h5py stempeg soundfile tqdm pyyaml librosa numba matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: Ignored the following versions that require a different python version: 0.0.12 Requires-Python >=3.5,<3.8
ERROR: Could not find a version that satisfies the requirement pickle5>=0.0.12 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11)
ERROR: No matching distribution found for pickle5>=0.0.12
  Using cached stempeg-0.2.6-py3-none-any.whl.metadata (9.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.2/963.2 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [stempeg]


In [9]:
# Extract data.zip into the repo root so you get: /content/splitify/hdf5s, /indexes, ...
!test -f "{DATA_ZIP_IN_DRIVE}" && echo "Found data.zip" || (echo "Missing data.zip at: {DATA_ZIP_IN_DRIVE}" && exit 1)
!unzip -q "{DATA_ZIP_IN_DRIVE}" -d "{REPO_DIR}"
!ls -la "{REPO_DIR}"


Found data.zip
replace /content/splitify/__MACOSX/._checkpoints? [y]es, [n]o, [A]ll, [N]one, [r]ename: total 72
drwxr-xr-x 10 root root 4096 Mar 16 13:49 .
drwxr-xr-x  1 root root 4096 Mar 16 12:49 ..
drwxr-xr-x  2 root root 4096 Mar 16 12:49 checkpoints
-rw-r--r--  1 root root  198 Mar 16 12:49 config.json
drwxr-xr-x  8 root root 4096 Mar 16 12:49 .git
-rw-r--r--  1 root root  411 Mar 16 12:49 .gitignore
drwxr-xr-x  3 root root 4096 Mar 16 12:49 hdf5s
drwxr-xr-x  3 root root 4096 Mar 16 13:49 indexes
-rw-r--r--  1 root root 8568 Mar 16 12:49 inference.py
drwxr-xr-x  4 root root 4096 Mar 16 12:49 learning
drwxr-xr-x  3 root root 4096 Mar 16 12:49 __MACOSX
drwxr-xr-x  2 root root 4096 Mar 16 12:49 notebooks
drwxr-xr-x  3 root root 4096 Mar 16 12:49 preprocessing
-rw-r--r--  1 root root 5608 Mar 16 12:49 README.md
-rw-r--r--  1 root root  440 Mar 16 12:49 requirements.txt


In [10]:
import os

# Ensure the current directory is the repo directory for subsequent commands that might rely on it.
# Also, explicitly provide absolute paths for python script execution for robustness.
os.chdir(REPO_DIR)

# Regenerate indexes on Colab so hdf5 paths match this filesystem
!python "{REPO_DIR}/preprocessing/index.py" \
  --workspace "{REPO_DIR}" \
  --config_yaml "{REPO_DIR}/preprocessing/configs/sr=44100,vocals-bass-drums-other.yaml" \
  --split train

Processing source: vocals, hdf5_dir=/content/splitify/hdf5s/musdb18/sr=44100,chn=2/train
Traceback (most recent call last):
  File "/content/splitify/preprocessing/index.py", line 83, in <module>
    create_indexes(args.workspace, args.config_yaml, args.split)
  File "/content/splitify/preprocessing/index.py", line 40, in create_indexes
    with h5py.File(hdf5_path, "r") as hf:
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 555, in __init__
    fid = make_fid(name, mode, userblock_size, fapl, fcpl, swmr=swmr)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 232, in make_fid
    fid = h5f.open(name, flags, fapl=fapl)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "h5py/h5f.pyx", line 106, i

Let's first inspect the contents of the `hdf5s` directory to confirm the files exist.

In [11]:
!ls -l "{REPO_DIR}/hdf5s/musdb18/sr=44100,chn=2/train"

total 27460088
-rw-r--r-- 1 root root  280276070 Sep  2  2025 'A Classic Education - NightOwl.h5'
-rw-r--r-- 1 root root  306816757 Sep  2  2025 "Actions - Devil's Words.h5"
-rw-r--r-- 1 root root  259596052 Sep  2  2025 'Actions - One Minute Smile.h5'
-rw-r--r-- 1 root root  269257881 Sep  2  2025 'Actions - South Of The Water.h5'
-rw-r--r-- 1 root root  265164841 Sep  2  2025 'Aimee Norwich - Child.h5'
-rw-r--r-- 1 root root  598064814 Sep  2  2025 'Alexander Ross - Goodbye Bolero.h5'
-rw-r--r-- 1 root root  708799955 Sep  2  2025 'Alexander Ross - Velvet Curtain.h5'
-rw-r--r-- 1 root root  306000903 Sep  2  2025 'Angela Thomas Wade - Milk Cow Blues.h5'
-rw-r--r-- 1 root root  351070601 Sep  2  2025 'ANiMAL - Clinic A.h5'
-rw-r--r-- 1 root root  261128634 Sep  2  2025 'ANiMAL - Rockshow.h5'
-rw-r--r-- 1 root root  374034953 Sep  2  2025 'Atlantis Bound - It Was My Fault For Waiting.h5'
-rw-r--r-- 1 root root  340132171 Sep  2  2025 'Auctioneer - Our Future Faces.h5'
-rw-r--r-- 1 root

Now, let's try to open one of the HDF5 files directly to check for corruption.

In [14]:
import h5py
import os

hdf5_file_path = os.path.join(REPO_DIR, "hdf5s/musdb18/sr=44100,chn=2/train", "Faces On Film - Waiting For Ga.h5") # Targeting the file identified as truncated

try:
    with h5py.File(hdf5_file_path, "r") as hf:
        print(f"Successfully opened {hdf5_file_path}. Dataset keys: {list(hf.keys())}")
except OSError as e:
    print(f"Error opening HDF5 file: {hdf5_file_path}")
    print(f"Error message: {e}")
    print("This confirms the HDF5 file is likely corrupted or incomplete. Please ensure your `data.zip` in Google Drive is valid and complete, and re-run the unzipping step.")
except FileNotFoundError:
    print(f"Error: HDF5 file not found at {hdf5_file_path}. Please check the path and file existence.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Error opening HDF5 file: /content/splitify/hdf5s/musdb18/sr=44100,chn=2/train/Faces On Film - Waiting For Ga.h5
Error message: Unable to synchronously open file (truncated file: eof = 28573696, sblock->base_addr = 0, stored_eof = 420748006)
This confirms the HDF5 file is likely corrupted or incomplete. Please ensure your `data.zip` in Google Drive is valid and complete, and re-run the unzipping step.


In [ ]:
# Train multi-stem model
# Start with a conservative batch size; increase if you don't hit CUDA OOM.
%cd "{REPO_DIR}/learning/multistem"

!python train.py \
  --workspace "{REPO_DIR}/checkpoints" \
  --index_pkl "{REPO_DIR}/indexes/musdb18/train/sr=44100,vocals-bass-drums-other.pkl" \
  --epochs 20 \
  --batch_size 2 \
  --lr 1e-3


In [ ]:
# Copy checkpoints to Drive so you don't lose them when Colab resets
!mkdir -p /content/drive/MyDrive/splitify_checkpoints
!cp -r "{REPO_DIR}/checkpoints"/* /content/drive/MyDrive/splitify_checkpoints/ || true
!ls -la /content/drive/MyDrive/splitify_checkpoints | tail
